# mva_09-クラスタ分析(2)

---

## **要点まとめ**

**Part Ⅲ：凝集度の最適化**

あらかじめ決めた数 $K$ にグループを分ける「非階層型（K-means法）」を学びました。今回は、**「階層型クラスタ分析 (Hierarchical Clustering)」** を学習します。
これは、個々のデータを最も似ているもの同士で少しずつ併合していき、最終的に一つの大きな塊になるまでの過程を **デンドログラム（樹形図）** として可視化する手法です。

### **1. 階層型クラスタ分析のアルゴリズム（凝集型）**

処理のイメージをつかむために、以下の単純な3つのデータ点（A, B, C）を例に考えてみましょう。



最も基本的な **凝集型 (Agglomerative)** の手順は以下の通りです。

1. **初期状態**:
全データ $n$ 個を、それぞれ $1$ 個のデータからなる $n$ 個のクラスタとみなす。
    - 【具体例】 数直線上の3つの点 $x_A=0$, $x_B=2$,  $x_C=5$
    - $\{A\}, \{B\}, \{C\}$ という3つのバラバラのクラスタからスタートします。

2. **距離計算**: すべてのクラスタ間の「距離（非類似度）」を計算し、**距離行列 (Distance Matrix)** を作成する。
    - 【初期の距離行列】
      | | A | B | C |
      |:--:|:--:|:--:|:--:|
      | A | 0 | 2 | 5 |
      | B | 2 | 0 | 3 |
      | C | 5 | 3 | 0 |

2. **併合 (Merge)**: 距離行列の中で 最も距離が近い（似ている）2つのクラスタ を探し、それらを合体させて1つの新しいクラスタにする。
    - 例：表の中で最も値が小さい（似ている）のは $A$と$B$の距離「2」 です。
    - $\rightarrow$ $A$と$B$を併合 し、新しいグループ $\{A, B\}$ を作ります。

3. **繰り返し**: クラスタ数が $1$ つになるまで、Step 2-3 を繰り返す。
    - 次は $\{A, B\}$ グループと $C$ の距離を計算し直して、最終的にすべてが合体するまで続けます。

この「併合の歴史」を記録し、木構造の図（デンドログラム）として表現します。

<br>


### **2. クラスタ間の距離の測り方（併合方式）**

点と点の距離はユークリッド距離などで定義できますが、「点の集まり（クラスタ）」と「点の集まり（クラスタ）」の距離をどう定義するかによって、形成されるクラスタの形状が変わります。代表的な手法は以下の通りです。

| 手法名 | 距離の定義  | 特徴 |
 | ----- | ----- | ----- |  
| **最短距離法**  (Single Linkage) | 最も近いデータ同士の距離 | 計算が簡単。細長いクラスタができやすい（鎖状効果）。|
| **最長距離法**  (Complete Linkage) | 最も遠いデータ同士の距離 | 凝集した（球状の）クラスタができやすい。外れ値に敏感。 |
| **ウォード法**  (Ward's Method) | $併合による「凝集度」の悪化量$  | **感度が良く、バランスの良い分類ができるため最も標準的に使われる。** |


<br>

### **3. ウォード法の視点：情報損失の最小化**

数ある併合方式の中で、なぜ ウォード法 (Ward's method) が標準的に使われるのでしょうか。それは、ウォード法が単なる「距離」ではなく、これまでの学習と同様に **「凝集度の最適化」** に着目しているからです。

#### **(1) クラスタ内の「情報量」の定義**

あるクラスタ $C$ が持っている「情報の豊かさ（＝まとまりの悪さ）」を、クラスタ内の **誤差平方和 (Sum of Squared Errors, SSE)** で定義し、これを目的関数 $J(C)$ とします。これはK-means法で最小化しようとした目的関数と同じものです。

$$J(C) = \sum_{\mathbf{x} \in C} ||\mathbf{x} - \mathbf{\bar{x}}_C||^2$$

- $\mathbf{x}$: クラスタ $C$ に含まれる各データ点ベクトル
- $\mathbf{\bar{x}}_C$: クラスタ $C$ の重心（平均ベクトル） $\frac{1}{|C|}\sum \mathbf{x}$

この $J(C)$ は、データが重心の周りにどれだけ散らばっているかを表しており、**「クラスタ内部の分散」** に比例する量です。

#### **(2) 併合による「情報の損失（コスト）」**

ここで、2つのクラスタ $C_A$ と $C_B$ を併合して、新しいクラスタ $C_{new}$ ($= C_A \cup C_B$) を作ることを考えます。
併合すると、重心が統合されるため、個々のデータから見た中心までの距離（ズレ）は必ず増加します。この増加分を **「情報損失量 $\Delta J$」** と定義し、これを2つのクラスタ間の距離とみなします。

$$\Delta J = J(C_{new}) - \left( J(C_A) + J(C_B) \right)$$

- $J(C_{new})$: 併合後の新しい誤差平方和
- $J(C_A) + J(C_B)$: 併合前の誤差平方和の合計

ウォード法は、すべてのペアの中で この **$\Delta J$ が最も小さくなるペア** を選んで併合する手法です。

#### **(3) 重心間距離との関係**

上記の定義式を展開すると、$\Delta J$ は以下の非常にシンプルな式で表せることが知られています。

$$\Delta J = \frac{n_A \cdot n_B}{n_A + n_B} ||\mathbf{\bar{x}}_A - \mathbf{\bar{x}}_B||^2$$

- $n_A, n_B$: クラスタ $A, B$ のデータ数
- $||\mathbf{\bar{x}}_A - \mathbf{\bar{x}}_B||^2$: 2つの重心間のユークリッド距離の2乗

この式の意味すること：
- **重心が近いほど良い**: 単純に中心同士が近いペアは $\Delta J$ が小さくなります。
- **データ数が少ないほど良い**: $n_A, n_B$ が小さい（小さなクラスタ同士）ほど係数が小さくなり、優先的に選ばれます。

結果として、ウォード法は **「サイズが小さく、かつ重心が近いペア」** を優先的にまとめる性質を持ちます。これにより、サイズが揃った、球状の凝集度の高いクラスタ が形成されやすくなります。また、ウォード法は本来複雑な「分散（全データの散らばり）」の変化を扱う指標ですが、数式変形によって **「重心同士のユークリッド距離」と「データ数（重み）」だけの簡単な式**に帰着できるため、計算コストを劇的に下げる（**高速化する**）ことができます。

<br>

**補足**: 重心間距離との関係は以下のように確認できます：
- [mva_09-補足資料：ウォード法の数理的導出.ipynb](https://colab.research.google.com/drive/1WZT85Gz6Ncm4M14cBhWh30IT29EhOnor?usp=sharing)
<br>

### **4. デンドログラムからのクラスタ数 $K$ の決定**

階層型クラスタリングの最大の利点は、**分析後に** デンドログラムを見てクラスタ数 $K$ を決められる点です。どこで切るのが適切かは、**「枝の長さ（縦軸の高さ）」** に注目します。

- **縦軸の意味**: 結合された瞬間の「非類似度（距離）」を表します。
- **長い枝**: 距離が離れた（似ていない）グループ同士を無理やり結合したことを意味します。

**【判断基準】** デンドログラムを下から見ていき、「急激に距離（縦軸）がジャンプしている箇所」 を探します。
枝が長く伸びている部分は、「その間、結合が起きなかった（＝安定したクラスタが存在していた）」ことを示唆しています。したがって、**最も長い縦線（ジャンプ）を横切るようにカット** すると、自然なクラスタ構造が得られることが多いです。

<br>



## **Python 演習課題**

### **mva_09-A**

✅ **目的**: 階層型クラスタリングにおける「距離行列の計算」と「併合による更新」のプロセスを手計算で追体験し、デンドログラムがどのように構築されるかを数理的に理解します。また、Pythonを用いてその併合プロセスをアニメーションで確認しましょう。

#### 🖊 **【数理理解】手計算による併合プロセスの体験**

以下の単純なデータ（4点）を用いて、**最短距離法（Single Linkage）** によるクラスタ併合がどう進むか計算してみましょう。
最短距離法は、「異なるグループの中で、最も近いメンバー同士の距離」をグループ間の距離とするシンプルな方法です。


【データ点】
$A(0, 0),\quad B(2, 0),\quad C(0, 3),\quad D(0, 4)$

**Step 0. 初期の距離行列**

まず、すべての点同士の距離（ユークリッド距離）を計算します。

|  | A | B | C | D |
| ----- | ----- | ----- | ----- | ----- |
| **A** | 0 | 2.0 | 3.0 | 4.0 |
| **B** | 2.0 | 0 | $\approx 3.6$ | $\approx 4.5$ |
| **C** | 3.0 | $\approx 3.6$ | 0 | **1.0** |
| **D** | 4.0 | $\approx 4.5$ | **1.0** | 0 |

<br>

**Step 1. 最小距離の探索と併合**

距離行列の中で、0以外の 最小値は「1.0」 です。これは点 C と D のペアです。
$\rightarrow$ CとDを併合 し、新しいクラスタ {C, D} を作ります。

<br>

**Step 2. 距離行列の更新**

新しくできたクラスタ {C, D} と、残りの点 A, B との距離を計算し直します。
今回は「最短距離法」なので、グループ内の最も近いメンバーとの距離を採用します。

- {C, D} と A の距離: $d(A, C)=3.0$, $d(A, D)=4.0$ なので、近い方の 3.0 を採用。
- {C, D} と B の距離: $d(B, C)\approx 3.6$, $d(B, D)\approx 4.5$ なので、近い方の 3.6 を採用。

|  | A | B | {C, D} |
| ----- | ----- | ----- | ----- |
| **A** | 0 | **2.0** | 3.0 |
| **B** | **2.0** | 0 | 3.6 |
| **{C, D}** | 3.0 | 3.6 | 0 |

<br>

**Step 3. 次の併合**

更新された行列の中で、最小値は 「2.0」 です。これは点 A と B のペアです。
$\rightarrow$ AとBを併合 し、新しいクラスタ {A, B} を作ります。

これでデータは、グループ {A, B} と グループ {C, D} の2つになりました。最後にこの2つが距離3.0で結合され、すべての併合が完了します。

<br>

#### **具体例で確認してみよう！**

以下のコードを実行して、この学習が進んでいく様子をアニメーションで確認しましょう。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ==========================================
# mva_09-A: 階層型クラスタリングの併合プロセス可視化
# ==========================================

# 1. データの定義 (手計算と同じデータ)
# A(0,0), B(2,0), C(0,3), D(0,4)
labels = ['A', 'B', 'C', 'D']
data = np.array([
    [0, 0],
    [2, 0],
    [0, 3],
    [0, 4]
])

# 2. クラスタリングの実行 (最短距離法: single)
# 手計算の例に合わせて 'single' (Single Linkage) を使用します
Z = linkage(data, method='single')

# --- 併合の履歴 (Linkage Matrix) を表示 ---
print("--- Linkage Matrix (併合の履歴) ---")
print("idx1, idx2, distance, count")
print(Z)
print("-" * 30)

# 再帰的にクラスタに含まれる元のデータ点のインデックスを取得する関数
def get_cluster_indices(cluster_idx, Z, n_samples):
    if cluster_idx < n_samples:
        return [cluster_idx]
    else:
        # Zの行 (cluster_idx - n_samples) を参照
        z_idx = cluster_idx - n_samples
        left = int(Z[z_idx, 0])
        right = int(Z[z_idx, 1])
        return get_cluster_indices(left, Z, n_samples) + get_cluster_indices(right, Z, n_samples)

# 3. アニメーションの作成
# 散布図上で、併合されたクラスタ同士を線で結んでいく様子を描画します

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 左側：散布図の初期設定（毎回クリアして再描画するため、枠組み設定のみ行う）
def setup_scatter_ax():
    ax1.clear()
    ax1.set_xlim(-1, 3)
    ax1.set_ylim(-1, 5)
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.set_title("Scatter Plot: Clustering Process")

# 右側：デンドログラム（静止画として最終形を表示しておく）
dendrogram(Z, labels=labels, ax=ax2)
ax2.set_title("Result: Dendrogram")

# アニメーション内で使用する閾値ラインの変数
threshold_line = None

# アニメーション更新関数
def update(frame):
    global threshold_line

    # 軸の初期化
    setup_scatter_ax()

    # --- デンドログラムの閾値ライン更新 ---
    # 前のフレームのラインがあれば削除
    if threshold_line is not None:
        threshold_line.remove()
        threshold_line = None

    n_samples = len(data)
    max_step = len(Z)
    steps_to_draw = min(frame, max_step)

    # 現在のステップに対応する距離にラインを引く
    if steps_to_draw > 0:
        # 最新の結合の距離を取得
        latest_merge_idx = steps_to_draw - 1
        current_dist = Z[latest_merge_idx, 2]

        # 右側のデンドログラムに水平線を描画
        threshold_line = ax2.axhline(y=current_dist, color='r', linestyle='--', alpha=0.8, linewidth=1.5)

    # --- 散布図の更新ロジック ---

    # 配色パレットの準備
    cmap = plt.get_cmap("tab10")

    # 初期状態のデータ点の色
    point_colors = [cmap(i) for i in range(n_samples)]

    # 色の更新: 現在のフレームまでに発生した併合を再現
    for i in range(steps_to_draw):
        new_cluster_idx = n_samples + i
        new_color = cmap(n_samples + i)
        indices = get_cluster_indices(new_cluster_idx, Z, n_samples)
        for pt_idx in indices:
            point_colors[pt_idx] = new_color

    # 1. データ点のプロット
    ax1.scatter(data[:, 0], data[:, 1], s=100, c=point_colors, zorder=5)

    # ラベルの表示
    for i, txt in enumerate(labels):
        ax1.annotate(txt, (data[i, 0]+0.1, data[i, 1]+0.1), fontsize=14, fontweight='bold')

    # 2. 結合線の描画
    step_text = f"Step {frame}: "
    if frame == 0:
        step_text += "Initial State"
    elif frame > max_step:
        step_text = "Final Result"

    current_step_idx = frame - 1

    for i in range(steps_to_draw):
        idx1, idx2 = int(Z[i, 0]), int(Z[i, 1])
        dist = Z[i, 2]

        # 重心を計算して線を引く
        points1_idx = get_cluster_indices(idx1, Z, n_samples)
        points2_idx = get_cluster_indices(idx2, Z, n_samples)

        p1 = np.mean(data[points1_idx], axis=0)
        p2 = np.mean(data[points2_idx], axis=0)

        # 線を引く (赤色で結合を示す)
        ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2, alpha=0.8)

        # 最新の結合のテキスト更新
        if i == current_step_idx:
            step_text += f"Merge Cluster {idx1} & {idx2} (Dist: {dist:.2f})"

    ax1.set_title(step_text, fontsize=12)
    return ax1,

# アニメーション生成
frames = len(Z) + 2
ani = FuncAnimation(fig, update, frames=frames, interval=1500, repeat=True)

plt.tight_layout()
# Colab/Jupyterで表示するための設定
plt.close() # 静的プロットを抑制
display(HTML(ani.to_jshtml()))

### **mva_09-B**
✅  **目的**: Pythonのライブラリを用いて階層型クラスタリングを実装し、手法（最短距離法・ウォード法）による結果の違いを視覚的に理解します。


#### 💻 **【実装・応用】 ライブラリによる階層クラスタリング**

階層型クラスタリングは、``scipy`` ライブラリの ``linkage`` 関数などを用いることで効率的に計算できます。
詳細な仕様については、以下の公式ドキュメントを参照してください。

- ``scipy`` 公式ドキュメント: ``scipy.cluster.hierarchy.linkage``


#### **具体例で確認してみよう！**

演習Aと同じ4つのデータ点 $A, B, C, D$ を用いますが、ここでは手計算ではなく ``scipy`` ライブラリを使って一瞬で計算を行います。
特に、**ウォード法 (Ward's method)** を適用した場合、最短距離法とは結合の順番や距離（高さ）が変わることに注目してください。

- 最短距離法 (Single): 鎖のように繋がりやすい。
- ウォード法 (Ward): まとまり（分散）を意識して結合するため、バランスの良いツリーになりやすい。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

# ==========================================
# mva_09-B: 手法によるデンドログラムの比較
# ==========================================

# 1. データの定義 (演習Aと同じ4点)
labels = ['A', 'B', 'C', 'D']
data = np.array([
    [0, 0],
    [2, 0],
    [0, 3],
    [0, 4]
])

print("--- データ座標 ---")
print(data)

# 2. 比較する手法のリスト
# single: 最短距離法, complete: 最長距離法, ward: ウォード法
methods = ['single', 'complete', 'ward']
titles = ['Single Linkage (Nearest)', 'Complete Linkage (Farthest)', "Ward's Method (Variance)"]

plt.figure(figsize=(15, 5))

for i, (method, title) in enumerate(zip(methods, titles)):
    # クラスタリングの実行 (結合行列 Z の作成)
    Z = linkage(data, method=method)

    # デンドログラムの描画
    plt.subplot(1, 3, i+1)
    dendrogram(Z, labels=labels)
    plt.title(title)
    plt.xlabel('Data Points')
    plt.ylabel('Distance')
    plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# --- 考察のヒント ---
# 最短距離法(Single)では、距離行列で見た通り A-B(dist=2) と C-D(dist=1) が結合されたあと、
# その2グループが dist=3 で結合されました。
#
# ウォード法(Ward)ではどうでしょうか？
# ウォード法での縦軸は「重心間の距離」だけでなく「データ数」も考慮した値になるため、
# Singleよりも値が大きく出ることが一般的です。